## 1. Environment Setup and Library Imports

In [ ]:
# Upgrade pip
%pip install --upgrade pip

# 1. Core Data Science & Statistics
%pip install "numpy<2.0.0" "pandas>=2.0.0" "scipy>=1.10.0" "statsmodels>=0.14.0"

# 2. Visualization
%pip install "matplotlib>=3.7.0" "seaborn>=0.12.0"

# 3. Machine Learning & Gradient Boosting
%pip install "scikit-learn>=1.3.0" "xgboost>=2.0.0" "catboost>=1.2.0" "lightgbm>=4.0.0"

# 4. Geospatial Processing
%pip install "rasterio>=1.3.0" "xarray>=2023.10.0" "rioxarray>=0.15.0" "geopandas>=0.14.0"

print("All required libraries have been successfully installed into the current kernel.")

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import jarque_bera

# Machine Learning & Evaluation Metrics
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, roc_curve, auc, precision_score,
    recall_score, f1_score, mean_squared_error,
    cohen_kappa_score, brier_score_loss
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

# Machine Learning Models (9 Classifiers)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# Geospatial Libraries
import rasterio
from rasterio.enums import Resampling
import xarray as xr
import rioxarray
import geopandas as gpd
from shapely.geometry import mapping

plt.rcParams['figure.figsize'] = (10, 6)
sns.set(style="whitegrid")
warnings.filterwarnings('ignore')

## 2. Data Loading and Data Preprocessing

In [ ]:
# Load 2018 dataset
df = pd.read_csv('features.csv')
df['flood'] = df['flood'].astype(int)
df = df.dropna()

print(f"Initial Dataset Shape: {df.shape}")

print("\n--- Jarque-Bera Normality Test Results ---")
jb_results = {}
for col in df.columns:
    if col != 'flood':
        stat, p = jarque_bera(df[col])
        jb_results[col] = {'JB Stat': stat, 'p-value': p, 'Normal': p > 0.05}
display(pd.DataFrame(jb_results).T)

print("\n--- Multicollinearity Analysis (VIF & Tolerance) ---")
continuous_vars = [
    "curvenumber90", "rainfalldepth90", "spi90", "twi90",
    "streamdensity90", "distancetostream90", "slope90", "dem90"
]

X_const = add_constant(df[continuous_vars])
vif_data = pd.DataFrame()
vif_data["Variable"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
vif_data["Tolerance"] = 1 / vif_data["VIF"]
display(vif_data)

In [ ]:
num_cols = [
    'rainfalldepth90', 'streamdensity90', 'distancetostream90',
    'slope90', 'dem90', 'curvenumber90', 'twi90', 'spi90'
]
cat_cols = ['lulc90', 'profilecurvatureclassified90', 'aspectclassified90']

# 1. Winsorization (1st and 99th Percentiles) to handle extreme outliers
lower_bounds = df[num_cols].quantile(0.01)
upper_bounds = df[num_cols].quantile(0.99)
df[num_cols] = df[num_cols].clip(lower=lower_bounds, upper=upper_bounds, axis=1)

# 2. Categorical Encoding (One-Hot)
df[cat_cols] = df[cat_cols].astype(int)
df = pd.get_dummies(df, columns=cat_cols, drop_first=False).astype(float)

X = df.drop('flood', axis=1)
y = df['flood']

# 3. Train-Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Robust Scaling
scaler = RobustScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])
train_cols = X_train.columns

# Save the fitted scaler for spatial inference and temporal validation
MODEL_DIR = "Saved_Models"
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(scaler, os.path.join(MODEL_DIR, "standard_scaler.pkl"))

# 5. Load and Preprocess 2019 Temporal Holdout Dataset
DATASET_2019_PATH = 'features_2019.csv'
df_2019 = pd.read_csv(DATASET_2019_PATH)
df_2019['flood'] = df_2019['flood'].astype(int)
df_2019 = df_2019.dropna()

X_2019 = df_2019.drop('flood', axis=1)
y_2019 = df_2019['flood']

X_2019[cat_cols] = X_2019[cat_cols].astype(int)
X_2019 = pd.get_dummies(X_2019, columns=cat_cols, drop_first=False).astype(float)
X_2019 = X_2019.reindex(columns=train_cols, fill_value=0)
X_2019[num_cols] = scaler.transform(X_2019[num_cols])

print(f"Preprocessing complete.")
print(f" -> 2018 Training Set Shape:   {X_train.shape}")
print(f" -> 2018 Validation Set Shape: {X_test.shape}")
print(f" -> 2019 Holdout Set Shape:    {X_2019.shape}")

## 3. Machine Learning Model Training

In [ ]:
# 3.1 Logistic Regression
log_reg = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
log_reg.fit(X_train, y_train)
joblib.dump(log_reg, os.path.join(MODEL_DIR, "LogisticRegression_model.pkl"))
print("Logistic Regression trained and saved.")

In [ ]:
# 3.2 Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_split=40,
    min_samples_leaf=20, n_jobs=-1, random_state=42
)
rf_model.fit(X_train, y_train)
joblib.dump(rf_model, os.path.join(MODEL_DIR, "RandomForest_model.pkl"))
print("Random Forest trained and saved.")

In [ ]:
# 3.3 XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=4, gamma=2.0,
    reg_lambda=10.0, subsample=0.7, colsample_bytree=0.7,
    random_state=42, eval_metric="logloss"
)
xgb_model.fit(X_train, y_train)
joblib.dump(xgb_model, os.path.join(MODEL_DIR, "XGBoost_model.pkl"))
print("XGBoost trained and saved.")

In [ ]:
# 3.4 CatBoost
catb_model = CatBoostClassifier(
    iterations=300, learning_rate=0.05, depth=4, l2_leaf_reg=15,
    loss_function='Logloss', eval_metric='AUC', random_state=42, verbose=False
)
catb_model.fit(X_train, y_train)
joblib.dump(catb_model, os.path.join(MODEL_DIR, "CatBoost_model.pkl"))
print("CatBoost trained and saved.")

In [ ]:
# 3.5 LightGBM
lgb_model = LGBMClassifier(
    n_estimators=200, learning_rate=0.05, num_leaves=15, max_depth=4,
    min_child_samples=40, reg_lambda=5.0, subsample=0.7,
    colsample_bytree=0.7, random_state=42, verbose=-1
)
lgb_model.fit(X_train, y_train)
joblib.dump(lgb_model, os.path.join(MODEL_DIR, "LightGBM_model.pkl"))
print("LightGBM trained and saved.")

In [ ]:
# 3.6 Support Vector Machine (RBF)
svm_model = SVC(kernel='rbf', C=2.0, gamma='scale', probability=True, random_state=42)
svm_model.fit(X_train, y_train)
joblib.dump(svm_model, os.path.join(MODEL_DIR, "SVM_model.pkl"))
print("SVM trained and saved.")

In [ ]:
# 3.7 Decision Tree
dtree_model = DecisionTreeClassifier(
    max_depth=10, min_samples_split=40, min_samples_leaf=20, random_state=42
)
dtree_model.fit(X_train, y_train)
joblib.dump(dtree_model, os.path.join(MODEL_DIR, "DecisionTree_model.pkl"))
print("Decision Tree trained and saved.")

In [ ]:
# 3.8 Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)
joblib.dump(nb_model, os.path.join(MODEL_DIR, "NaiveBayes_model.pkl"))
print("Naive Bayes trained and saved.")

In [ ]:
# 3.9 K-Nearest Neighbors (KNN)
knn_model = KNeighborsClassifier(n_neighbors=40, weights='uniform', metric='minkowski')
knn_model.fit(X_train, y_train)
joblib.dump(knn_model, os.path.join(MODEL_DIR, "KNN_model.pkl"))
print("KNN trained and saved.")

# Register all 9 trained models for evaluation and spatial mapping
loaded_models = {
    "LogisticRegression": log_reg,
    "RandomForest": rf_model,
    "XGBoost": xgb_model,
    "CatBoost": catb_model,
    "LightGBM": lgb_model,
    "SVM": svm_model,
    "DecisionTree": dtree_model,
    "NaiveBayes": nb_model,
    "KNN": knn_model
}
print(f"\nRegistered {len(loaded_models)} models: {list(loaded_models.keys())}")

## 4. Model Evaluation

### 4.1 ROC-AUC Curve Evaluation 

In [ ]:
sorted_model_names = sorted(loaded_models.keys())

fig, axes = plt.subplots(3, 3, figsize=(18, 18))
fig.suptitle(
    'ROC Curves: 2018 Flood Training, 2018 Validation, and 2019 Flood Event Validation',
    fontsize=20, y=1.02
)

for idx, name in enumerate(sorted_model_names):
    model = loaded_models[name]
    ax = axes.flatten()[idx]

    try:
        # Predict probabilities for 2018 Train, 2018 Test, and 2019 Holdout
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]
        y_2019_prob = model.predict_proba(X_2019)[:, 1]

        # Compute ROC and AUC for 2018 Training
        fpr_train, tpr_train, _ = roc_curve(y_train, y_train_prob)
        roc_auc_train = auc(fpr_train, tpr_train)

        # Compute ROC and AUC for 2018 Validation
        fpr_test, tpr_test, _ = roc_curve(y_test, y_test_prob)
        roc_auc_test = auc(fpr_test, tpr_test)

        # Compute ROC and AUC for 2019 Event Validation
        fpr_2019, tpr_2019, _ = roc_curve(y_2019, y_2019_prob)
        roc_auc_2019 = auc(fpr_2019, tpr_2019)

        # Plot curves
        ax.plot(fpr_train, tpr_train, color='black', lw=2, label=f'Train (AUC = {roc_auc_train:.4f})')
        ax.plot(fpr_test, tpr_test, color='red', lw=2, label=f'2018 Validation (AUC = {roc_auc_test:.4f})')
        ax.plot(fpr_2019, tpr_2019, color='blue', lw=2, label=f'2019 Validation (AUC = {roc_auc_2019:.4f})')
        ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--')

        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate')
        ax.set_ylabel('True Positive Rate')
        ax.set_title(name, fontsize=14, fontweight='bold')
        ax.legend(loc="lower right", fontsize=10)
        ax.grid(alpha=0.3, linestyle='--')

    except Exception as e:
        ax.set_title(f"{name} (Failed)")
        print(f"Failed to plot ROC for {name}. Error: {e}")

plt.tight_layout()
plt.show()

### 4.2 Calcularing the Evaluation Metrics

In [ ]:
def evaluate_models_on_dataset(models_dict, X_eval, y_eval, prefix=""):
    """
    Computes classification and probabilistic calibration metrics across all models.
    """
    eval_results = {}
    for model_name, model_obj in models_dict.items():
        try:
            y_prob = model_obj.predict_proba(X_eval)[:, 1]
            y_pred = (y_prob > 0.5).astype(int)
            tn, fp, fn, tp = confusion_matrix(y_eval, y_pred).ravel()

            eval_results[model_name] = {
                f'{prefix}ROC_AUC': roc_auc_score(y_eval, y_prob),
                f'{prefix}Accuracy': accuracy_score(y_eval, y_pred),
                f'{prefix}Precision': precision_score(y_eval, y_pred, zero_division=0),
                f'{prefix}Recall': recall_score(y_eval, y_pred, zero_division=0),
                f'{prefix}Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
                f'{prefix}F1_Score': f1_score(y_eval, y_pred, zero_division=0),
                f'{prefix}Kappa': cohen_kappa_score(y_eval, y_pred),
                f'{prefix}RMSE': np.sqrt(mean_squared_error(y_eval, y_prob)),
                f'{prefix}Brier': brier_score_loss(y_eval, y_prob)
            }
        except Exception as e:
            print(f"Error evaluating {model_name}: {e}")

    return pd.DataFrame(eval_results).T.sort_values(by=f'{prefix}ROC_AUC', ascending=False)

# 1. 2018 Validation (Test) Metrics
print("--- 2018 Validation Set Metrics ---")
results_2018_df = evaluate_models_on_dataset(loaded_models, X_test, y_test, prefix="2018_")
display(results_2018_df)

export_path_2018 = "Model_Evaluation_Metrics_2018_Validation.csv"
results_2018_df.to_csv(export_path_2018, float_format='%.4f')
print(f"2018 Validation Metrics exported to: {export_path_2018}\n")

# 2. 2019 Temporal Holdout Validation Metrics
print("--- 2019 Temporal Validation Metrics ---")
results_2019_df = evaluate_models_on_dataset(loaded_models, X_2019, y_2019, prefix="2019_")
display(results_2019_df)

export_path_2019 = "Model_Evaluation_Metrics_2019_Holdout.csv"
results_2019_df.to_csv(export_path_2019, float_format='%.4f')
print(f"2019 Temporal Validation Metrics exported to: {export_path_2019}")

# 3. Visualize 2019 Temporal Validation ROC-AUC
plt.figure(figsize=(10, 6))
sns.barplot(x=results_2019_df['2019_ROC_AUC'], y=results_2019_df.index, palette='magma')
plt.title('Temporal Validation ROC-AUC Score (2019 Dataset)')
plt.xlabel('ROC-AUC')
plt.xlim(0.5, 1.0)
plt.grid(alpha=0.3, axis='x', linestyle='--')
plt.tight_layout()
plt.show()

## 5. Seasonal Risk Index Computation & Analysis

In [ ]:
# --- 1. CONFIGURATION ---
KERALA_CRS = "EPSG:7781"
GEOJSON_FILE = "kerala.geojson"
ALIGNED_LAYERS_DIR = "aligned_layers"
GRID_TEMPLATE_FILE = os.path.join(ALIGNED_LAYERS_DIR, "dem90.tif")
INPUT_DIR = "seasonal_factors"
OUTPUT_DIR="Seasonal Risk Index"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEASONAL_WEIGHTS = {
    'pre':  {'rain': 0.4, 'cape': 0.4, 'asm': 0.2},
    'post': {'rain': 0.4, 'cape': 0.1, 'asm': 0.5}
}

# Pre-monsoon and Post-monsoon dynamic layers (Peak monsoon rainfall is used directly in the ML models)
FILES = {
    'rain': {
        'pre': os.path.join(INPUT_DIR, 'rain_pre.tif'),
        'post': os.path.join(INPUT_DIR, 'rain_post.tif')
    },
    'cape': {
        'pre': os.path.join(INPUT_DIR, 'cape_pre.tif'),
        'post': os.path.join(INPUT_DIR, 'cape_post.tif')
    },
    'asm': {
        'pre': os.path.join(INPUT_DIR, 'asm_pre.tif'),
        'post': os.path.join(INPUT_DIR, 'asm_post.tif')
    }
}

# --- 2. ROBUST RASTER LOADER ---
def open_clean_raster(path):
    """
    Reads GeoTIFF as raw pixels, converting masked values to np.nan
    and avoiding coordinate metadata conflicts in xarray.
    """
    with rasterio.open(path) as src:
        masked_data = src.read(1, masked=True)
        data = masked_data.filled(np.nan)

        height, width = data.shape
        transform = src.transform

        x_coords = np.linspace(
            transform.c + transform.a / 2,
            transform.c + transform.a * (width - 0.5),
            width
        )
        y_coords = np.linspace(
            transform.f + transform.e / 2,
            transform.f + transform.e * (height - 0.5),
            height
        )

        da = xr.DataArray(
            data,
            coords={"y": y_coords, "x": x_coords},
            dims=("y", "x")
        )
        da.rio.write_crs(src.crs, inplace=True)
        da.rio.write_transform(transform, inplace=True)
        return da

# --- 3. GLOBAL NORMALIZATION LOGIC ---
def load_and_normalize_globally(file_dict, reference_raster):
    loaded_rasters = {}
    print(f"  > Loading seasons: {list(file_dict.keys())}...")

    for season, filename in file_dict.items():
        if not os.path.exists(filename):
            raise FileNotFoundError(f"Missing file: {filename}")

        da = open_clean_raster(filename)
        if da.rio.crs is None:
            da.rio.write_crs(KERALA_CRS, inplace=True)

        da = da.rio.reproject_match(reference_raster)
        da_clean = da.drop_vars(["spatial_ref", "band"], errors="ignore")
        loaded_rasters[season] = da_clean

    vals = [d.values for d in loaded_rasters.values()]
    g_min = np.nanmin(vals)
    g_max = np.nanmax(vals)
    print(f"    > Global Stats | Min: {g_min:.4f} | Max: {g_max:.4f}")

    norm_rasters = {}
    for season, da in loaded_rasters.items():
        norm = (da - g_min) / (g_max - g_min + 1e-6)
        norm_rasters[season] = norm.clip(min=0, max=1)

    return norm_rasters

# --- 4. EXECUTE SRI GENERATION ---
if not os.path.exists(GRID_TEMPLATE_FILE):
    raise FileNotFoundError(f"Base template grid '{GRID_TEMPLATE_FILE}' not found.")

print(f"Loading Base Spatial Grid Template: {GRID_TEMPLATE_FILE}...")
ref_grid = rioxarray.open_rasterio(GRID_TEMPLATE_FILE, masked=True).squeeze()
if ref_grid.rio.crs is None:
    ref_grid.rio.write_crs(KERALA_CRS, inplace=True)

if not os.path.exists(GEOJSON_FILE):
    raise FileNotFoundError(f"Boundary file '{GEOJSON_FILE}' not found.")

print(f"Loading Clipping Boundary: {GEOJSON_FILE}...")
kerala_gdf = gpd.read_file(GEOJSON_FILE)
if kerala_gdf.crs != ref_grid.rio.crs:
    kerala_gdf = kerala_gdf.to_crs(ref_grid.rio.crs)

print("\n--- Processing Seasonal Rainfall ---")
rain_norm = load_and_normalize_globally(FILES['rain'], ref_grid)

print("\n--- Processing Seasonal CAPE ---")
cape_norm = load_and_normalize_globally(FILES['cape'], ref_grid)

print("\n--- Processing Seasonal ASM ---")
asm_norm = load_and_normalize_globally(FILES['asm'], ref_grid)

print("\n--- Generating Clipped Seasonal Risk Index (SRI) TIFs ---")
for season in ['pre', 'post']:
    w = SEASONAL_WEIGHTS[season]
    seasonal_index = (
        (rain_norm[season] * w['rain']) +
        (cape_norm[season] * w['cape']) +
        (asm_norm[season] * w['asm'])
    )
    seasonal_index.rio.write_crs(ref_grid.rio.crs, inplace=True)

    clipped_index = seasonal_index.rio.clip(
        kerala_gdf.geometry.apply(mapping),
        kerala_gdf.crs,
        drop=False,
        invert=False
    )
    
    out_name = f"seasonal_risk_index_{season}.tif"
    output_path = os.path.join(OUTPUT_DIR, out_name)
    clipped_index.rio.to_raster( output_path)
    print(f"  -> Saved: {out_name}")

print("\nSRI generation complete.")

## 6. Flood Susceptibility Map Generation & Analysis

### 6.1 Peak Monsoon Susceptibility Maps

In [ ]:
BASE_MAPS_DIR = "Generated_Maps"
os.makedirs(BASE_MAPS_DIR, exist_ok=True)

def generate_susceptibility_map(model, model_name, output_filename):
    print(f"Generating Peak Monsoon susceptibility map for {model_name}...")

    stream_density_path = (
        f"{ALIGNED_LAYERS_DIR}/streamdensity90.tif"
        if os.path.exists(f"{ALIGNED_LAYERS_DIR}/streamdensity90.tif")
        else f"{ALIGNED_LAYERS_DIR}/streamdensity9090.tif"
    )

    raster_paths = {
        "curvenumber90": f"{ALIGNED_LAYERS_DIR}/curvenumber90.tif",
        "rainfalldepth90": f"{ALIGNED_LAYERS_DIR}/rainfalldepth90.tif",
        "spi90": f"{ALIGNED_LAYERS_DIR}/spi90.tif",
        "twi90": f"{ALIGNED_LAYERS_DIR}/twi90.tif",
        "lulc90": f"{ALIGNED_LAYERS_DIR}/lulc90.tif",
        "streamdensity90": stream_density_path,
        "distancetostream90": f"{ALIGNED_LAYERS_DIR}/distancetostream90.tif",
        "profilecurvatureclassified90": f"{ALIGNED_LAYERS_DIR}/profilecurvatureclassified90.tif",
        "aspectclassified90": f"{ALIGNED_LAYERS_DIR}/aspectclassified90.tif",
        "slope90": f"{ALIGNED_LAYERS_DIR}/slope90.tif",
        "dem90": f"{ALIGNED_LAYERS_DIR}/dem90.tif"
    }

    ref_key = "dem90"
    with rasterio.open(raster_paths[ref_key]) as ref:
        meta = ref.meta.copy()
        shape = (ref.height, ref.width)

    data = {}
    for name, path in raster_paths.items():
        with rasterio.open(path) as src:
            if (src.height, src.width) != shape:
                arr = src.read(1, out_shape=shape, resampling=Resampling.nearest)
            else:
                arr = src.read(1)
            arr = arr.astype('float64')
            if src.nodata is not None:
                arr[arr == src.nodata] = np.nan
            data[name] = arr

    pred_df = pd.DataFrame({name: arr.ravel() for name, arr in data.items()})
    valid_mask = ~pred_df.isna().any(axis=1)
    pred_valid = pred_df.loc[valid_mask].copy()

    saved_scaler = joblib.load(os.path.join(MODEL_DIR, "standard_scaler.pkl"))
    pred_valid[num_cols] = saved_scaler.transform(pred_valid[num_cols])
    pred_encoded = pd.get_dummies(pred_valid, columns=cat_cols)
    pred_encoded = pred_encoded.reindex(columns=train_cols, fill_value=0)

    probs_valid = model.predict_proba(pred_encoded)[:, 1]

    probs_full = np.full(pred_df.shape[0], np.nan)
    probs_full[valid_mask] = probs_valid
    prob_raster = probs_full.reshape(shape)

    meta.update(dtype=rasterio.float32, count=1, nodata=np.nan, compress='lzw')
    output_path = os.path.join(BASE_MAPS_DIR, output_filename)
    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(prob_raster.astype(rasterio.float32), 1)

    print(f" -> Saved: {output_path}")

print("--- Starting Batch Peak Monsoon Susceptibility Map Generation ---")
for model_name, model_obj in loaded_models.items():
    output_map_name = f"prob_flood_{model_name}.tif"
    try:
        generate_susceptibility_map(model_obj, model_name, output_map_name)
    except Exception as e:
        print(f"Failed to generate map for {model_name}. Error: {e}")

### 6.2 Seasonal Susceptibility Maps

In [ ]:
OUTPUT_FOLDER = "Final_Seasonal_Maps"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("--- Applying Seasonal Risk Index (SRI) to Peak Monsoon Maps for All 9 Models ---")

indices = {}
for season in ['pre', 'post']:
    index_file = f"seasonal_index_{season}.tif"
    if os.path.exists(index_file):
        indices[season] = rioxarray.open_rasterio(index_file, masked=True).squeeze()
    else:
        raise FileNotFoundError(f"{index_file} not found. Run Section 6 first.")

for model_name in loaded_models.keys():
    base_map_path = os.path.join(BASE_MAPS_DIR, f"prob_flood_{model_name}.tif")
    if not os.path.exists(base_map_path):
        print(f"Base map for {model_name} not found. Skipping...")
        continue

    print(f"\nProcessing seasonal susceptibility maps for {model_name}...")
    model_da = rioxarray.open_rasterio(base_map_path, masked=True).squeeze()

    # Export Peak Monsoon susceptibility map to Final_Dynamic_Maps
    peak_path = os.path.join(OUTPUT_FOLDER, f"Dynamic_{model_name}_peak.tif")
    model_da.astype("float32").rio.to_raster(peak_path)
    print(f"  -> Saved Peak Monsoon: {peak_path}")

    # Compute and export Pre-Monsoon and Post-Monsoon maps using SRI
    for season in ['pre', 'post']:
        index_da = indices[season]
        if model_da.shape != index_da.shape:
            index_aligned = index_da.rio.reproject_match(model_da)
        else:
            index_aligned = index_da

        dynamic_risk = model_da * index_aligned
        out_path = os.path.join(OUTPUT_FOLDER, f"Dynamics_{model_name}_{season}.tif")
        dynamic_risk.astype("float32").rio.to_raster(out_path)
        print(f"  -> Saved {season.capitalize()}-Monsoon: {out_path}")

print("\nSeasonal susceptibility map generation complete.")

### 6.3 Multi-Model SRI Weight Sensitivity Analysis

In [ ]:
TARGET_MODELS = ["XGBoost", "SVM", "KNN"]

scenarios = [
    {'Scenario': 'PM1 - Proposed', 'Season': 'pre',  'Rain': 0.40, 'ASM': 0.20, 'CAPE': 0.40},
    {'Scenario': 'PM2',            'Season': 'pre',  'Rain': 0.50, 'ASM': 0.10, 'CAPE': 0.40},
    {'Scenario': 'PM3',            'Season': 'pre',  'Rain': 0.30, 'ASM': 0.30, 'CAPE': 0.40},
    {'Scenario': 'PM4',            'Season': 'pre',  'Rain': 0.40, 'ASM': 0.30, 'CAPE': 0.30},
    {'Scenario': 'PM5',            'Season': 'pre',  'Rain': 0.40, 'ASM': 0.10, 'CAPE': 0.50},
    {'Scenario': 'PM6 - Equal',    'Season': 'pre',  'Rain': 0.33, 'ASM': 0.33, 'CAPE': 0.33},
    {'Scenario': 'PO1 - Proposed', 'Season': 'post', 'Rain': 0.40, 'ASM': 0.50, 'CAPE': 0.10},
    {'Scenario': 'PO2',            'Season': 'post', 'Rain': 0.40, 'ASM': 0.40, 'CAPE': 0.20},
    {'Scenario': 'PO3',            'Season': 'post', 'Rain': 0.50, 'ASM': 0.40, 'CAPE': 0.10},
    {'Scenario': 'PO4',            'Season': 'post', 'Rain': 0.30, 'ASM': 0.60, 'CAPE': 0.10},
    {'Scenario': 'PO5 - Equal',    'Season': 'post', 'Rain': 0.33, 'ASM': 0.33, 'CAPE': 0.33}
]

sensitivity_results = []
print("Executing Multi-Model Spatial Sensitivity Analysis...")

for model_name in TARGET_MODELS:
    base_map_path = os.path.join(BASE_MAPS_DIR, f"prob_flood_{model_name}.tif")
    if not os.path.exists(base_map_path):
        print(f"  ! Warning: Cannot find {base_map_path}. Skipping {model_name}.")
        continue

    model_da = rioxarray.open_rasterio(base_map_path, masked=True).squeeze()

    for s in scenarios:
        season = s['Season']
        dynamic_sri = (
            (rain_norm[season] * s['Rain']) +
            (asm_norm[season] * s['ASM']) +
            (cape_norm[season] * s['CAPE'])
        )

        if model_da.rio.crs is None:
            model_da.rio.write_crs(KERALA_CRS, inplace=True)
        if dynamic_sri.rio.crs is None:
            dynamic_sri.rio.write_crs(KERALA_CRS, inplace=True)

        if model_da.shape != dynamic_sri.shape:
            dynamic_sri = dynamic_sri.rio.reproject_match(model_da)

        dynamic_risk_values = model_da.values * dynamic_sri.values
        valid_pixels = dynamic_risk_values.flatten()
        valid_pixels = valid_pixels[~np.isnan(valid_pixels)]

        total_valid = len(valid_pixels)
        if total_valid == 0:
            continue

        counts, _ = np.histogram(valid_pixels, bins=5, range=(valid_pixels.min(), valid_pixels.max()))
        percentages = (counts / total_valid) * 100

        sensitivity_results.append({
            'Model': model_name,
            'Season': 'Pre-monsoon' if season == 'pre' else 'Post-monsoon',
            'Weight Set': s['Scenario'],
            'Rainfall': s['Rain'],
            'ASM': s['ASM'],
            'CAPE': s['CAPE'],
            'Very Low (%)': percentages[0],
            'Low (%)': percentages[1],
            'Medium (%)': percentages[2],
            'High (%)': percentages[3],
            'Very High (%)': percentages[4]
        })

df_sensitivity = pd.DataFrame(sensitivity_results)
format_cols = ['Very Low (%)', 'Low (%)', 'Medium (%)', 'High (%)', 'Very High (%)']
df_sensitivity[format_cols] = df_sensitivity[format_cols].round(2)

export_csv = "MultiModel_SRI_Sensitivity_Table.csv"
df_sensitivity.to_csv(export_csv, index=False)
print(f"Sensitivity analysis exported to: {export_csv}")
display(df_sensitivity)

### 6.4 Susceptibility Class Area Percentage Analysis

In [ ]:
CLASS_LABELS = ['Very Low', 'Low', 'Medium', 'High', 'Very High']

def get_class_percentages(raster_path):
    """
    Reads a susceptibility raster, masks NoData, bins valid pixels into 5 equal
    intervals, and returns the percentage of total area per class.
    """
    if not os.path.exists(raster_path):
        print(f"  [!] Warning: File not found -> {raster_path}")
        return [np.nan] * 5

    with rasterio.open(raster_path) as src:
        data = src.read(1, masked=True)
        valid_pixels = data.compressed()
        valid_pixels = valid_pixels[~np.isnan(valid_pixels)]

        total_valid = len(valid_pixels)
        if total_valid == 0:
            return [0.0] * 5

        counts, _ = np.histogram(valid_pixels, bins=5, range=(valid_pixels.min(), valid_pixels.max()))
        return (counts / total_valid) * 100

master_results = []

for model_name in loaded_models.keys():
    map_paths = {
        "Peak Monsoon": os.path.join(OUTPUT_FOLDER, f"Dynamic_{model_name}_peak.tif"),
        "Pre-Monsoon": os.path.join(OUTPUT_FOLDER, f"Dynamics_{model_name}_pre.tif"),
        "Post-Monsoon": os.path.join(OUTPUT_FOLDER, f"Dynamics_{model_name}_post.tif")
    }

    model_results = {
        label: get_class_percentages(path)
        for label, path in map_paths.items()
    }

    df_model = pd.DataFrame(model_results, index=CLASS_LABELS)

    for label in df_model.columns:
        master_results.append({
            "Model": model_name,
            "Season / Framework": label,
            "Very Low (%)": df_model.loc["Very Low", label],
            "Low (%)": df_model.loc["Low", label],
            "Medium (%)": df_model.loc["Medium", label],
            "High (%)": df_model.loc["High", label],
            "Very High (%)": df_model.loc["Very High", label]
        })

df_master = pd.DataFrame(master_results)
export_csv = "Master_Area_Percentage_Comparison.csv"
df_master.to_csv(export_csv, index=False, float_format='%.2f')

print(f"Master area percentage table exported to: {export_csv}")
display(df_master.round(2))

In [ ]:
# Consolidated Multi-Model Visualizations
df_long = pd.melt(
    df_master,
    id_vars=['Model', 'Season / Framework'],
    value_vars=['Very Low (%)', 'Low (%)', 'Medium (%)', 'High (%)', 'Very High (%)'],
    var_name='Susceptibility Class',
    value_name='Area Percentage (%)'
)
df_long['Susceptibility Class'] = df_long['Susceptibility Class'].str.replace(' (%)', '', regex=False)

# 1. Small Multiples Facet Grid
g = sns.catplot(
    data=df_long,
    x='Susceptibility Class',
    y='Area Percentage (%)',
    hue='Season / Framework',
    col='Model',
    col_wrap=3,
    kind='bar',
    palette='viridis',
    height=4.5,
    aspect=1.2,
    sharey=False,
    width=0.5
)
g.figure.suptitle('Seasonal Flood Susceptibility Area Distribution Across Models', fontsize=20, fontweight='bold', y=1.02)
g.set_titles("{col_name}", size=14, fontweight='bold')
g.set_axis_labels("", "Area Percentage (%)")
g.set_xticklabels(rotation=45)
plt.tight_layout()
g.savefig("Consolidated_FacetGrid_Models.png", dpi=300, bbox_inches='tight')
plt.show()

# 2. 100% Stacked Bar Chart
df_stacked = df_master.set_index(['Model', 'Season / Framework'])[
    ['Very Low (%)', 'Low (%)', 'Medium (%)', 'High (%)', 'Very High (%)']
]
colors = ['#1a9641', '#a6d96a', '#ffffbf', '#fdae61', '#d7191c']

fig, ax = plt.subplots(figsize=(18, 8))
df_stacked.plot(kind='bar', stacked=True, color=colors, width=0.4, ax=ax)

plt.title('Spatial Composition of Susceptibility Classes per Model and Season', fontsize=18, fontweight='bold')
plt.xlabel('Model and Season', fontsize=14)
plt.ylabel('Percentage of Total Area (%)', fontsize=14)
plt.ylim(0, 100)

labels = [f"{idx[0]}\n({idx[1]})" for idx in df_stacked.index]
ax.set_xticklabels(labels, rotation=90, fontsize=10)
plt.legend(title='Susceptibility Class', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig("Consolidated_StackedBar_Models.png", dpi=300, bbox_inches='tight')
plt.show()